# Weather Agent with LangChain and OpenAI API

This notebook demonstrates how to build a simple weather-getting agent using LangChain and OpenAI's API. The agent can fetch current weather and forecasts for any location using the Open-Meteo API (free, no key required).

## 1. Import Required Libraries

Import all necessary libraries for building the weather agent, including LangChain, OpenAI, and requests for API interactions.

In [ ]:
import os
import json
import requests
from typing import Optional
from dotenv import load_dotenv

from langchain.tools import Tool
from langchain.agents import initialize_agent, AgentType
from langchain_openai import ChatOpenAI

# Load environment variables from .env file
load_dotenv()

print("Libraries imported successfully!")

## 2. Set Up OpenAI API Key

Configure your OpenAI API key. You can set it in multiple ways:
- Create a `.env` file with `OPENAI_API_KEY=your_key_here`
- Set it directly as an environment variable
- Pass it when creating the ChatOpenAI instance

In [ ]:
# Check if OpenAI API key is set
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✓ OpenAI API key is configured")
    print(f"  API Key (masked): {api_key[:10]}...{api_key[-4:]}")
else:
    print("⚠ Warning: OPENAI_API_KEY not found in environment")
    print("  Please set it before running the agent")
    print("\n  Options:")
    print("  1. Create a .env file with: OPENAI_API_KEY=sk-...")
    print("  2. Or set as environment variable")

## 3. Define Weather Tool Functions

Create functions to fetch weather data from the Open-Meteo API (free, no API key required).

In [ ]:
def get_current_weather(location: str) -> str:
    """
    Get current weather for a location using Open-Meteo API.
    
    Args:
        location: City name (e.g., "London", "New York", "Tokyo")
    
    Returns:
        JSON string with current weather data
    """
    try:
        # Step 1: Geocode the location
        geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
        geocoding_params = {
            "name": location,
            "count": 1,
            "language": "en",
            "format": "json"
        }
        
        geo_response = requests.get(geocoding_url, params=geocoding_params, timeout=10)
        geo_response.raise_for_status()
        geo_data = geo_response.json()
        
        if not geo_data.get("results"):
            return json.dumps({
                "error": f"Location '{location}' not found",
                "status": "failed"
            })
        
        # Get coordinates
        location_data = geo_data["results"][0]
        latitude = location_data["latitude"]
        longitude = location_data["longitude"]
        name = location_data.get("name", location)
        country = location_data.get("country", "")
        
        # Step 2: Get weather data
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
            "temperature_unit": "celsius",
            "wind_speed_unit": "kmh",
            "timezone": "auto"
        }
        
        weather_response = requests.get(weather_url, params=weather_params, timeout=10)
        weather_response.raise_for_status()
        weather_data = weather_response.json()
        
        current = weather_data.get("current", {})
        
        # Map weather codes to descriptions
        weather_descriptions = {
            0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
            45: "Foggy", 48: "Foggy with rime", 51: "Light drizzle", 53: "Moderate drizzle",
            61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain", 71: "Slight snow",
            73: "Moderate snow", 75: "Heavy snow", 80: "Rain showers", 85: "Snow showers",
            95: "Thunderstorm", 96: "Thunderstorm with hail", 99: "Thunderstorm with heavy hail"
        }
        
        weather_code = current.get("weather_code", 0)
        description = weather_descriptions.get(weather_code, "Unknown")
        
        result = {
            "location": f"{name}, {country}",
            "temperature": f"{current.get('temperature_2m', 'N/A')}°C",
            "humidity": f"{current.get('relative_humidity_2m', 'N/A')}%",
            "wind_speed": f"{current.get('wind_speed_10m', 'N/A')} kmh",
            "weather": description,
            "status": "success"
        }
        
        return json.dumps(result, indent=2)
    
    except Exception as e:
        return json.dumps({
            "error": f"Error fetching weather: {str(e)}",
            "status": "failed"
        })


def get_weather_forecast(location: str) -> str:
    """
    Get 7-day weather forecast for a location.
    
    Args:
        location: City name
    
    Returns:
        JSON string with forecast data
    """
    try:
        # Geocode location
        geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"
        geocoding_params = {"name": location, "count": 1, "language": "en", "format": "json"}
        
        geo_response = requests.get(geocoding_url, params=geocoding_params, timeout=10)
        geo_response.raise_for_status()
        geo_data = geo_response.json()
        
        if not geo_data.get("results"):
            return json.dumps({"error": f"Location '{location}' not found", "status": "failed"})
        
        location_data = geo_data["results"][0]
        latitude = location_data["latitude"]
        longitude = location_data["longitude"]
        name = location_data.get("name", location)
        
        # Fetch forecast
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_params = {
            "latitude": latitude,
            "longitude": longitude,
            "daily": "temperature_2m_max,temperature_2m_min,weather_code,precipitation_sum",
            "temperature_unit": "celsius",
            "timezone": "auto",
            "forecast_days": 7
        }
        
        weather_response = requests.get(weather_url, params=weather_params, timeout=10)
        weather_response.raise_for_status()
        weather_data = weather_response.json()
        
        daily = weather_data.get("daily", {})
        
        weather_descriptions = {
            0: "Clear", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
            45: "Foggy", 51: "Drizzle", 61: "Rain", 71: "Snow", 80: "Showers", 95: "Thunderstorm"
        }
        
        forecast = []
        for i in range(len(daily.get("time", []))):
            forecast.append({
                "date": daily["time"][i],
                "max_temp": f"{daily['temperature_2m_max'][i]}°C",
                "min_temp": f"{daily['temperature_2m_min'][i]}°C",
                "weather": weather_descriptions.get(daily["weather_code"][i], "Unknown"),
                "precipitation": f"{daily['precipitation_sum'][i]}mm"
            })
        
        return json.dumps({"location": name, "forecast": forecast, "status": "success"}, indent=2)
    
    except Exception as e:
        return json.dumps({"error": f"Error fetching forecast: {str(e)}", "status": "failed"})

print("Weather tool functions defined!")

## 4. Create LangChain Tools

Wrap the weather functions as LangChain Tool objects that the agent can use.

In [ ]:
# Create LangChain Tool objects
weather_tool = Tool(
    name="get_current_weather",
    func=get_current_weather,
    description="Get current weather information for a location. Input should be a city name (e.g., 'London', 'New York')"
)

forecast_tool = Tool(
    name="get_weather_forecast",
    func=get_weather_forecast,
    description="Get a 7-day weather forecast for a location. Input should be a city name (e.g., 'London', 'New York')"
)

# List of all tools the agent can use
tools = [weather_tool, forecast_tool]

print(f"Created {len(tools)} tools:")
for tool in tools:
    print(f"  • {tool.name}: {tool.description}")

## 5. Initialize LLM and Create Agent

Initialize the OpenAI language model and create a ReAct (Reasoning + Acting) agent that can use the weather tools.

In [ ]:
# Initialize the OpenAI LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",  # or "gpt-4" for better results
    temperature=0,          # Use 0 for deterministic responses
    verbose=True
)

print(f"LLM initialized: {llm.model_name}\n")

# Create the agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print("✓ Weather Agent created and ready to use!")

## 6. Test Agent with Sample Query

Run the agent with a sample weather query to demonstrate its functionality.

In [ ]:
# Test with a sample query
query = "What's the current weather in London?"
print(f"Query: {query}\n")
print("="*60)

response = agent.run(query)

print("="*60)
print(f"\nAgent Response:\n{response}")

## 7. Test Agent with Multiple Queries

Test the agent with various weather-related queries to validate its performance.

In [ ]:
# Test multiple weather queries
test_queries = [
    "What's the weather like in Paris?",
    "Will it rain in Tokyo tomorrow?",
    "I need the 7-day forecast for New York",
    "Is it cold in Moscow right now?",
    "Get the weather for Sydney"
]

print("Testing agent with multiple queries:\n")

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*60}")
    print(f"Query {i}: {query}")
    print('='*60)
    
    try:
        response = agent.run(query)
        print(f"Response:\n{response}")
    except Exception as e:
        print(f"Error: {str(e)}")
    
    print()